Notebook pour le mémoire sur la détection de dégâts routiers. Le dataset RDD2022 (aliabdelmenam/rdd-2022) doit être ajouté en Input avant de lancer ce notebook. Accélérateur GPU requis pour les cellules d'entraînement.

In [1]:
!pip install ultralytics --quiet


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.7/46.7 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 53.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.7/88.7 kB 6.8 MB/s eta 0:00:00


In [2]:
import torch
print("GPU disponible :", torch.cuda.is_available())


GPU disponible : True


Filtrage du dataset pour ne garder que D20 (fissure alligator) et D40 (nid-de-poule), classes 2 et 4 dans ce mirror. Le mirror ne documente pas son mapping de classes, identifié en regardant les images annotées directement.

In [3]:
import os
import shutil

SOURCE_ROOT = "/kaggle/input/datasets/aliabdelmenam/rdd-2022/RDD_SPLIT"
OUTPUT_ROOT = "/kaggle/working/rdd_filtre"

CLASSES_A_GARDER = {2: 0, 4: 1}
NOMS_CLASSES = ["D20", "D40"]


def filtrer_split(nom_split):
    dossier_images_src = f"{SOURCE_ROOT}/{nom_split}/images"
    dossier_labels_src = f"{SOURCE_ROOT}/{nom_split}/labels"
    dossier_images_dst = f"{OUTPUT_ROOT}/images/{nom_split}"
    dossier_labels_dst = f"{OUTPUT_ROOT}/labels/{nom_split}"
    os.makedirs(dossier_images_dst, exist_ok=True)
    os.makedirs(dossier_labels_dst, exist_ok=True)

    nb_gardees = 0
    nb_total = 0

    for nom_label in os.listdir(dossier_labels_src):
        nb_total += 1
        chemin_label = f"{dossier_labels_src}/{nom_label}"
        lignes_gardees = []

        with open(chemin_label) as f:
            for ligne in f:
                if not ligne.strip():
                    continue
                parties = ligne.split()
                classe_id = int(parties[0])
                if classe_id in CLASSES_A_GARDER:
                    nouvelle_classe = CLASSES_A_GARDER[classe_id]
                    parties[0] = str(nouvelle_classe)
                    lignes_gardees.append(" ".join(parties))

        if not lignes_gardees:
            continue

        nom_image = nom_label.replace(".txt", ".jpg")
        chemin_image_src = f"{dossier_images_src}/{nom_image}"
        if not os.path.exists(chemin_image_src):
            continue

        shutil.copy(chemin_image_src, f"{dossier_images_dst}/{nom_image}")
        with open(f"{dossier_labels_dst}/{nom_label}", "w") as f:
            f.write("\n".join(lignes_gardees))
        nb_gardees += 1

    print(f"{nom_split} : {nb_gardees}/{nb_total} images gardées")


for split in ["train", "val"]:
    filtrer_split(split)

contenu_yaml = f"""train: {OUTPUT_ROOT}/images/train
val: {OUTPUT_ROOT}/images/val

nc: {len(NOMS_CLASSES)}
names: {NOMS_CLASSES}
"""
with open(f"{OUTPUT_ROOT}/data.yaml", "w") as f:
    f.write(contenu_yaml)

print(contenu_yaml)


train : 7464/26869 images gardées
val : 1558/5758 images gardées
train: /kaggle/working/rdd_filtre/images/train
val: /kaggle/working/rdd_filtre/images/val

nc: 2
names: ['D20', 'D40']



Quatre entraînements testés. Tout ce qui est écrit dans /kaggle/working reste disponible dans l'onglet Output après Save Version, pas besoin de le copier ailleurs pendant la session.

Baseline YOLOv8n, 15 epochs, pour vérifier que le pipeline de données marche. mAP50 obtenu : 0.520.

In [4]:
from ultralytics import YOLO

model = YOLO("yolov8n.pt")

resultats = model.train(
    data="/kaggle/working/rdd_filtre/data.yaml",
    epochs=15,
    imgsz=640,
    batch=16,
    project="/kaggle/working/runs_entrainement",
    name="baseline_D20_D40",
)


Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.
Ultralytics 8.4.159 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/rdd_filtre/data.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=15, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fractio

/usr/local/lib/python3.12/dist-packages/ray/train/_internal/session.py:676: UserWarning: `get_trial_id` is meant to only be called inside a function that is executed by a Tuner or Trainer. Returning `None`.
  warnings.warn(


       2/15      2.57G      1.948       2.53      1.852         17        640: 100% ━━━━━━━━━━━━ 467/467 6.9it/s 1:08
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 49/49 5.9it/s 8.3s
                   all       1558       2518      0.377      0.229      0.177     0.0654

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
       3/15      2.57G      1.933      2.424      1.838         35        640: 100% ━━━━━━━━━━━━ 467/467 6.9it/s 1:08
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 49/49 5.7it/s 8.5s
                   all       1558       2518      0.317      0.265      0.213     0.0813

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
       4/15      2.57G      1.898      2.348      1.817         19        640: 100% ━━━━━━━━━━━━ 467/467 6.8it/s 1:09
                 Class     Images  Instances      Box(P 

YOLOv8n complet, 80 epochs, imgsz=640. Modèle retenu pour le pipeline vidéo. mAP50 = 0.594 (D20 : 0.685, D40 : 0.504). Pas de surapprentissage détecté.

In [5]:
from ultralytics import YOLO

model = YOLO("yolov8n.pt")

resultats = model.train(
    data="/kaggle/working/rdd_filtre/data.yaml",
    epochs=80,
    imgsz=640,
    batch=16,
    project="/kaggle/working/runs_entrainement",
    name="entrainement_final_D20_D40",
    save_period=10,
)


Ultralytics 8.4.159 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/rdd_filtre/data.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=80, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=entrainement_final_D20_D40, nbs=64, nms=

YOLOv8s, même protocole. mAP50 = 0.610 (D20 : 0.708, D40 : 0.512), mieux mais inférence 2.8x plus lente (6.5ms contre 2.3ms par image).

In [6]:
from ultralytics import YOLO

model = YOLO("yolov8s.pt")

resultats = model.train(
    data="/kaggle/working/rdd_filtre/data.yaml",
    epochs=80,
    imgsz=640,
    batch=-1,
    project="/kaggle/working/runs_entrainement",
    name="yolov8s_D20_D40",
    save_period=10,
)


Ultralytics 8.4.159 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=-1, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/rdd_filtre/data.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=80, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8s.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=yolov8s_D20_D40, nbs=64, nms=None, opset

YOLOv8n avec résolution doublée (imgsz=960). Interrompu à l'epoch 49 sur 80 sur Colab par manque de quota GPU, relancé ici en entier.

In [7]:
from ultralytics import YOLO

model = YOLO("yolov8n.pt")

resultats = model.train(
    data="/kaggle/working/rdd_filtre/data.yaml",
    epochs=80,
    imgsz=960,
    batch=-1,
    project="/kaggle/working/runs_entrainement",
    name="yolov8n_imgsz960_D20_D40",
    save_period=10,
)


Ultralytics 8.4.159 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=-1, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/rdd_filtre/data.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=80, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=960, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=yolov8n_imgsz960_D20_D40, nbs=64, nms=No

Chargement d'un des modèles entraînés dans cette session pour la suite.

In [8]:
from ultralytics import YOLO

CHEMIN_MODELE = "/kaggle/working/runs_entrainement/entrainement_final_D20_D40/weights/best.pt"
model = YOLO(CHEMIN_MODELE)
print("Classes :", model.names)


Classes : {0: 'D20', 1: 'D40'}


Pipeline vidéo : découpage en frames, détection, mesure du FPS, vidéo annotée en sortie. Nécessite une vidéo ajoutée en Input ou uploadée dans /kaggle/working.

In [9]:
import cv2, time, json

CHEMIN_VIDEO_ENTREE = "/kaggle/working/video_test.mp4"
CHEMIN_VIDEO_SORTIE = "/kaggle/working/video_annotee.mp4"
CHEMIN_JSON_DETECTIONS = "/kaggle/working/detections.json"
SEUIL_CONFIANCE = 0.25

noms_classes = model.names

capture = cv2.VideoCapture(CHEMIN_VIDEO_ENTREE)
if not capture.isOpened():
    raise RuntimeError(f"Impossible d'ouvrir la vidéo : {CHEMIN_VIDEO_ENTREE}")

largeur = int(capture.get(cv2.CAP_PROP_FRAME_WIDTH))
hauteur = int(capture.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps_source = capture.get(cv2.CAP_PROP_FPS)
nb_frames_total = int(capture.get(cv2.CAP_PROP_FRAME_COUNT))
print(f"Vidéo source : {largeur}x{hauteur}, {fps_source:.1f} FPS, {nb_frames_total} frames")

codec = cv2.VideoWriter_fourcc(*"mp4v")
ecrivain = cv2.VideoWriter(CHEMIN_VIDEO_SORTIE, codec, fps_source, (largeur, hauteur))

toutes_les_detections = []
numero_frame = 0
temps_debut = time.time()

while True:
    lu_correctement, frame = capture.read()
    if not lu_correctement:
        break

    resultats = model.predict(frame, conf=SEUIL_CONFIANCE, verbose=False)
    resultat = resultats[0]

    for boite in resultat.boxes:
        classe_id = int(boite.cls[0])
        score_confiance = float(boite.conf[0])
        x1, y1, x2, y2 = boite.xyxy[0].tolist()
        toutes_les_detections.append({
            "frame": numero_frame,
            "temps_video_sec": round(numero_frame / fps_source, 2),
            "classe": noms_classes[classe_id],
            "confiance": round(score_confiance, 3),
            "boite": [round(x1), round(y1), round(x2), round(y2)],
        })

    frame_annotee = resultat.plot()
    ecrivain.write(frame_annotee)

    numero_frame += 1
    if numero_frame % 100 == 0:
        print(f"  {numero_frame}/{nb_frames_total} frames traitées...")

temps_fin = time.time()
capture.release()
ecrivain.release()

duree_totale = temps_fin - temps_debut
fps_inference = numero_frame / duree_totale if duree_totale > 0 else 0

print("Résumé du traitement")
print(f"Frames traitées      : {numero_frame}")
print(f"Durée totale         : {duree_totale:.1f} sec")
print(f"FPS d'inférence      : {fps_inference:.2f} images/sec")
print(f"Détections trouvées  : {len(toutes_les_detections)}")

compteur_classes = {}
for d in toutes_les_detections:
    compteur_classes[d["classe"]] = compteur_classes.get(d["classe"], 0) + 1
print(f"Répartition par classe : {compteur_classes}")

with open(CHEMIN_JSON_DETECTIONS, "w") as f:
    json.dump(toutes_les_detections, f, indent=2, ensure_ascii=False)


RuntimeError: Impossible d'ouvrir la vidéo : /kaggle/working/video_test.mp4

Inférence par lot sur un dossier de vidéos et images de Cotonou (ajouté en Input ou uploadé dans /kaggle/working).

In [ ]:
import cv2, time, json, os
from pathlib import Path

DOSSIER_ENTREE = "/kaggle/working/donnees_cotonou"
DOSSIER_SORTIE = "/kaggle/working/resultats_cotonou"
SEUIL_CONFIANCE = 0.25

EXTENSIONS_VIDEO = {".mp4", ".mov", ".avi", ".mkv"}
EXTENSIONS_IMAGE = {".jpg", ".jpeg", ".png"}

os.makedirs(DOSSIER_SORTIE, exist_ok=True)
noms_classes = model.names
resume_global = []


def traiter_image(chemin_fichier, nom_fichier):
    image = cv2.imread(str(chemin_fichier))
    if image is None:
        print(f"  Impossible de lire : {nom_fichier}")
        return
    temps_debut = time.time()
    resultats = model.predict(image, conf=SEUIL_CONFIANCE, verbose=False)
    duree = time.time() - temps_debut
    resultat = resultats[0]
    detections = [{"classe": noms_classes[int(b.cls[0])], "confiance": round(float(b.conf[0]), 3)} for b in resultat.boxes]
    image_annotee = resultat.plot()
    chemin_sortie = f"{DOSSIER_SORTIE}/{Path(nom_fichier).stem}_annotee.jpg"
    cv2.imwrite(chemin_sortie, image_annotee)
    resume_global.append({"fichier": nom_fichier, "type": "image", "nb_detections": len(detections),
                           "detections": detections, "temps_traitement_sec": round(duree, 3)})
    print(f"  {nom_fichier} : {len(detections)} détection(s)")


def traiter_video(chemin_fichier, nom_fichier):
    capture = cv2.VideoCapture(str(chemin_fichier))
    if not capture.isOpened():
        print(f"  Impossible d'ouvrir : {nom_fichier}")
        return
    largeur = int(capture.get(cv2.CAP_PROP_FRAME_WIDTH))
    hauteur = int(capture.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps_source = capture.get(cv2.CAP_PROP_FPS) or 25
    chemin_sortie = f"{DOSSIER_SORTIE}/{Path(nom_fichier).stem}_annotee.mp4"
    ecrivain = cv2.VideoWriter(chemin_sortie, cv2.VideoWriter_fourcc(*"mp4v"), fps_source, (largeur, hauteur))

    numero_frame = 0
    toutes_detections = []
    temps_debut = time.time()
    while True:
        lu, frame = capture.read()
        if not lu:
            break
        resultats = model.predict(frame, conf=SEUIL_CONFIANCE, verbose=False)
        resultat = resultats[0]
        for b in resultat.boxes:
            toutes_detections.append({"frame": numero_frame, "classe": noms_classes[int(b.cls[0])],
                                       "confiance": round(float(b.conf[0]), 3)})
        ecrivain.write(resultat.plot())
        numero_frame += 1
    duree = time.time() - temps_debut
    capture.release()
    ecrivain.release()

    fps_inference = numero_frame / duree if duree > 0 else 0
    resume_global.append({
        "fichier": nom_fichier, "type": "video", "nb_frames": numero_frame,
        "nb_detections": len(toutes_detections), "fps_inference": round(fps_inference, 2),
        "repartition_classes": {c: sum(1 for d in toutes_detections if d["classe"] == c) for c in noms_classes.values()},
    })
    print(f"  {nom_fichier} : {len(toutes_detections)} détections sur {numero_frame} frames, {fps_inference:.1f} FPS")


fichiers = sorted(os.listdir(DOSSIER_ENTREE))
print(f"{len(fichiers)} fichiers trouvés dans {DOSSIER_ENTREE}")

for nom_fichier in fichiers:
    chemin_fichier = Path(DOSSIER_ENTREE) / nom_fichier
    extension = chemin_fichier.suffix.lower()
    if extension in EXTENSIONS_VIDEO:
        print(f"[VIDÉO] {nom_fichier}")
        traiter_video(chemin_fichier, nom_fichier)
    elif extension in EXTENSIONS_IMAGE:
        print(f"[IMAGE] {nom_fichier}")
        traiter_image(chemin_fichier, nom_fichier)
    else:
        print(f"[IGNORÉ] {nom_fichier} (extension non reconnue)")

with open(f"{DOSSIER_SORTIE}/resume_global.json", "w") as f:
    json.dump(resume_global, f, indent=2, ensure_ascii=False)

print(f"Fichiers traités : {len(resume_global)}")
total_detections = sum(r["nb_detections"] for r in resume_global)
print(f"Détections totales toutes sources confondues : {total_detections}")
